In [ ]:
# ============================================================
# INDIA FLOOD PREDICTION — REALISTIC TRAINING PIPELINE
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# STEP 1: Load Merged Dataset
# Upload india_flood_merged_final.csv to Colab
# ─────────────────────────────────────────────
df = pd.read_csv("india_flood_merged_final.csv")
print("✅ Dataset loaded. Shape:", df.shape)

# ─────────────────────────────────────────────
# STEP 2: Drop Fully-Null & Leaky Columns
# avg_severity is all null → drop it
# Rainfall_Anomaly_Z was used to CREATE Flood_Label
# → keeping it causes 100% accuracy (data leakage)
# ─────────────────────────────────────────────
df.drop(columns=['avg_severity', 'Event_x_Severity',
                 'Rainfall_Anomaly_Z', 'Rainfall_x_Anomaly',
                 'Anomaly_Squared'], inplace=True, errors='ignore')

# ─────────────────────────────────────────────
# STEP 3: Fill Remaining Nulls
# ─────────────────────────────────────────────
for col in ['ANNUAL', 'Rolling_3yr_Annual',
            'Annual_x_Monsoon', 'Monsoon_to_Annual_Ratio']:
    df[col].fillna(df[col].median(), inplace=True)

# ─────────────────────────────────────────────
# STEP 4: Define Features (No Leaky Columns)
# ─────────────────────────────────────────────
features = [
    'ANNUAL',
    'Monsoon_Rainfall',
    'Rolling_3yr_Annual',
    'flood_event_count',
    'avg_duration_days',
    'avg_fatalities',
    'dominant_cause_encoded',
    'Duration_x_Fatality',
    'Monsoon_to_Annual_Ratio',
]

X = df[features].fillna(0)
y = df['Flood_Label']

print("\n📊 Class Distribution:")
print(y.value_counts())

# ─────────────────────────────────────────────
# STEP 5: Time-Aware Train/Test Split
# Use last 20% of years as test (realistic evaluation)
# ─────────────────────────────────────────────
df_sorted = df.sort_values('YEAR').reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.80)

train_df = df_sorted.iloc[:split_idx]
test_df  = df_sorted.iloc[split_idx:]

X_train = train_df[features].fillna(0)
y_train = train_df['Flood_Label']
X_test  = test_df[features].fillna(0)
y_test  = test_df['Flood_Label']

print(f"\n✅ Train size: {len(X_train)} | Test size: {len(X_test)}")

# ─────────────────────────────────────────────
# STEP 6: Partial Oversample Minority Class
# (Avoids extreme imbalance without overfitting)
# ─────────────────────────────────────────────
train_combined = pd.concat([X_train, y_train], axis=1)
majority    = train_combined[train_combined['Flood_Label'] == 0]
minority    = train_combined[train_combined['Flood_Label'] == 1]
minority_up = resample(minority, replace=True,
                       n_samples=int(len(majority) * 0.4),
                       random_state=42)
train_bal   = pd.concat([majority, minority_up]).sample(frac=1, random_state=42)

X_train_bal = train_bal[features]
y_train_bal = train_bal['Flood_Label']

# ─────────────────────────────────────────────
# STEP 7: Train Random Forest
# (Constrained depth to prevent overfitting)
# ─────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train_bal, y_train_bal)
rf_acc = accuracy_score(y_test, rf.predict(X_test))

print(f"\n✅ Random Forest Accuracy : {rf_acc * 100:.2f}%")
print(classification_report(y_test, rf.predict(X_test)))

# ─────────────────────────────────────────────
# STEP 8: Train Gradient Boosting
# (Shallow trees + low learning rate = no overfitting)
# ─────────────────────────────────────────────
gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=42
)
gb.fit(X_train_bal, y_train_bal)
gb_acc = accuracy_score(y_test, gb.predict(X_test))

print(f"✅ Gradient Boosting Accuracy : {gb_acc * 100:.2f}%")
print(classification_report(y_test, gb.predict(X_test)))

# ─────────────────────────────────────────────
# STEP 9: Best Model Summary
# ─────────────────────────────────────────────
best = 'Random Forest' if rf_acc >= gb_acc else 'Gradient Boosting'
print(f"\n🏆 Best Model: {best} → {max(rf_acc, gb_acc) * 100:.2f}% Accuracy")

✅ Dataset loaded. Shape: (4116, 20)

📊 Class Distribution:
Flood_Label
0    3824
1     292
Name: count, dtype: int64

✅ Train size: 3292 | Test size: 824

✅ Random Forest Accuracy : 84.83%
              precision    recall  f1-score   support

           0       0.98      0.85      0.91       771
           1       0.27      0.79      0.40        53

    accuracy                           0.85       824
   macro avg       0.63      0.82      0.66       824
weighted avg       0.94      0.85      0.88       824

✅ Gradient Boosting Accuracy : 91.87%
              precision    recall  f1-score   support

           0       0.98      0.94      0.96       771
           1       0.42      0.68      0.52        53

    accuracy                           0.92       824
   macro avg       0.70      0.81      0.74       824
weighted avg       0.94      0.92      0.93       824


🏆 Best Model: Gradient Boosting → 91.87% Accuracy


In [ ]:
import joblib

# ─────────────────────────────────────────────
# STEP 10: Save Models
# ─────────────────────────────────────────────

# Save Random Forest
joblib.dump(rf, "random_forest_flood_model.pkl")

# Save Gradient Boosting
joblib.dump(gb, "gradient_boosting_flood_model.pkl")

print("\n💾 Models saved successfully!")


💾 Models saved successfully!


In [ ]:
import joblib
import os

# Create content folder explicitly (safe practice)
save_path = "/content/"
os.makedirs(save_path, exist_ok=True)

# ─────────────────────────────────────────────
# Save Models
# ─────────────────────────────────────────────
joblib.dump(rf, os.path.join(save_path, "random_forest_flood_model.pkl"))
joblib.dump(gb, os.path.join(save_path, "gradient_boosting_flood_model.pkl"))

# ─────────────────────────────────────────────
# Save Label Encoder
# ─────────────────────────────────────────────
joblib.dump(le, os.path.join(save_path, "label_encoder.pkl"))

# ─────────────────────────────────────────────
# Save Metadata
# ─────────────────────────────────────────────
metadata = {
    "features": features,
    "target": "Flood_Label",
    "rf_accuracy": rf_acc,
    "gb_accuracy": gb_acc,
    "best_model": best
}

joblib.dump(metadata, os.path.join(save_path, "metadata.pkl"))

print("✅ All files saved inside /content/")

✅ All files saved inside /content/
